## Step 3 — Tikhonov deconvolution with reflective boundaries

We keep the original RGB image resolution:

$$
512\times512\times3,
\qquad
n=512,
\qquad
N=n^2=262144.
$$

Let

$$
c\in\{R,G,B\}
$$

denote the colour channel.

The motion blur was generated with

$$
\texttt{angle}=0^\circ,
$$

therefore it is exactly horizontal. The same spatial blur acts independently on the red, green, and blue channels, without mixing different rows or colour channels.

### 2. Row-wise forward model

Let the $r$-th row of colour channel $c$ be represented as a column vector:

$$
f_{r,c}=
\begin{bmatrix}
f_{r,1,c}\\
f_{r,2,c}\\
\vdots\\
f_{r,512,c}
\end{bmatrix}
\in\mathbb{R}^{512}.
$$

Since the blur is horizontal, the corresponding degraded row is

$$
g_{r,c}=Bf_{r,c}+\epsilon_{r,c},
$$

where

$$
B\in\mathbb{R}^{512\times512}
$$

is the one-dimensional horizontal blur operator and $\epsilon_{r,c}$ is the noise vector.

The same matrix $B$ is used for every image row and for every RGB channel.

### 3. Periodic boundaries and Fourier-domain deconvolution

With periodic boundary conditions, horizontal convolution is described by a circulant matrix $C_{\mathrm{per}}$. The discrete Fourier transform diagonalizes this operator.

For each colour channel $c$,

$$
G_c(u)=H(u)F_c(u)+E_c(u).
$$

The direct inverse filter would be

$$
F_{\mathrm{inv},c}(u)
=
\frac{G_c(u)}{H(u)}.
$$

The Tikhonov-regularized inverse filter would be

$$
F_{\lambda,c}(u)
=
\frac{H^*(u)}{|H(u)|^2+\lambda}G_c(u).
$$

These Fourier-domain formulas are applied independently to the RGB channels and are valid only when the forward model assumes periodic boundary conditions.

### 4. Why the Fourier regularized inverse filter is not used

Our processed images were generated with reflective padding rather than periodic boundary conditions.

With periodic boundaries, the signal wraps around at the edges:

$$
f_{513}=f_1.
$$

With reflective boundaries, the signal is mirrored:

$$
[f_1,f_2,\ldots,f_{511},f_{512}]
\rightarrow
[\ldots,f_3,f_2,f_1,f_1,f_2,f_3,\ldots,f_{512},f_{512},f_{511},\ldots].
$$

Therefore, the actual blur operator is not circulant:

$$
B\neq C_{\mathrm{per}}.
$$

Using the simple Fourier-domain regularized inverse filter would solve the wrong inverse problem, because it would assume periodic boundaries instead of reflective ones.

### 5. Construction of the reflective blur matrix

The matrix $B$ is explicitly constructed to represent the same horizontal blur used to generate the processed images.

For a row $f_{r,c}$, the forward operation is

$$
f_{r,c}
\overset{P_{\mathrm{ref}}}{\longrightarrow}
\text{reflected padded row}
\overset{\mathcal{C}_{p_L}}{\longrightarrow}
\text{convolution with }p_L
\overset{R}{\longrightarrow}
\text{cropped output}.
$$

Thus,

$$
B=R\,\mathcal{C}_{p_L}\,P_{\mathrm{ref}}.
$$

Each column of $B$ is obtained by applying this operation to a basis vector:

$$
B_{:,j}
=
\text{reflective horizontal blur of }e_j.
$$

Hence, for every row and colour channel,

$$
g_{r,c}=Bf_{r,c}+\epsilon_{r,c}.
$$

The matrix $B$ is constructed only once and then reused for all rows and all RGB channels.

### 6. Full image model

For each colour channel, we stack all rows into one vector:

$$
f_c=
\begin{bmatrix}
f_{1,c}\\
f_{2,c}\\
\vdots\\
f_{512,c}
\end{bmatrix},
\qquad
g_c=
\begin{bmatrix}
g_{1,c}\\
g_{2,c}\\
\vdots\\
g_{512,c}
\end{bmatrix}.
$$

The forward model for each channel is

$$
g_c=Af_c+\epsilon_c,
$$

where

$$
A=
\begin{bmatrix}
B & 0 & \cdots & 0\\
0 & B & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & B
\end{bmatrix}
=
I_{512}\otimes B.
$$

If the three channels are stacked together,

$$
f_{\mathrm{RGB}}=
\begin{bmatrix}
f_R\\
f_G\\
f_B
\end{bmatrix},
\qquad
g_{\mathrm{RGB}}=
\begin{bmatrix}
g_R\\
g_G\\
g_B
\end{bmatrix},
$$

then

$$
g_{\mathrm{RGB}}
=
A_{\mathrm{RGB}}f_{\mathrm{RGB}}
+
\epsilon_{\mathrm{RGB}},
$$

with

$$
A_{\mathrm{RGB}}
=
I_3\otimes A.
$$

Neither $A$ nor $A_{\mathrm{RGB}}$ is constructed explicitly. Their block structure allows us to work through the smaller matrix $B$.

In [8]:
# importiamo le immagini clean e processed
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

data_path_clean = Path("/Users/pietro34/Documents/GitHub/Photodeblurring_IPProject/data/processed/clean")
data_path_processed= Path("/Users/pietro34/Documents/GitHub/Photodeblurring_IPProject/data/processed/blur+noise")

image_ids = [1, 2, 3]
noise_levels = [0.02, 0.05, 0.10]
noise_labels = ["0.02", "0.05", "0.10"]
clean_images = [
    plt.imread(data_path_clean / f"{i}.png")[..., :3]
    for i in image_ids
]

processed_images = [
    [
        plt.imread(data_path_processed / f"{i}_blur_noise{noise}.png")[..., :3]
        for noise in noise_labels
    ]
    for i in image_ids
]



In [20]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
scripts_path = project_root / "scripts"



if str(scripts_path) not in sys.path:
    sys.path.insert(0, str(scripts_path))

from kernel import motion_blur_psf
from operators import apply_motion_blur

In [22]:
def build_horizontal_reflective_blur_matrix(psf, image_width):
    """
    Build the 1D horizontal blur matrix B such that

        g_r = B @ f_r

    for every image row f_r, using the same reflective-padding
    operator adopted in operators.apply_motion_blur.
    """
    psf = np.asarray(psf, dtype=float)

    if psf.ndim != 2:
        raise ValueError("The PSF must be a two-dimensional array.")

    central_row = psf.shape[0] // 2
    off_axis_psf = np.delete(psf, central_row, axis=0)

    if not np.allclose(off_axis_psf, 0.0):
        raise ValueError(
            "The PSF is not strictly horizontal: it has non-zero "
            "coefficients outside its central row."
        )

    identity_image = np.eye(image_width, dtype=float)

    # apply_motion_blur(I) = B.T, hence B is obtained by transposition.
    return operators.apply_motion_blur(identity_image, psf).T

### Construction of the blur matrix $B$

The matrix $B$ is built by applying the same forward blur operator to all canonical basis vectors at once.

Let

$$
I_{512}=
\begin{bmatrix}
e_1^T\\
e_2^T\\
\vdots\\
e_{512}^T
\end{bmatrix},
$$

where $e_j$ is the $j$-th basis vector of $\mathbb{R}^{512}$.

The function `apply_motion_blur` is applied to the identity matrix using the motion PSF:

$$
\mathcal{A}_{\mathrm{ref}}(I_{512}).
$$

Each row of the identity matrix represents one basis vector. Therefore, the $j$-th row of the blurred result corresponds to the blurred version of $e_j^T$.

Since the row-wise forward model is written as

$$
g_r=Bf_r,
$$

while the image is stored with rows as row vectors, the blurred image satisfies

$$
G=FB^T.
$$

Hence,

$$
\mathcal{A}_{\mathrm{ref}}(I_{512})=B^T,
$$

and the blur matrix is obtained as

$$
B=
\left[
\mathcal{A}_{\mathrm{ref}}(I_{512})
\right]^T.
$$

This procedure constructs a matrix

$$
B\in\mathbb{R}^{512\times512}
$$

that reproduces exactly the same blur operator used in the generation of the processed images.

In [25]:
image_width = clean_images[0].shape[1]

length = 21
angle = 0

psf = kernel.motion_blur_psf(length, angle)

B = build_horizontal_reflective_blur_matrix(
    psf=psf,
    image_width=image_width
)

print("PSF shape:", psf.shape)
print("B shape:", B.shape)


PSF shape: (21, 21)
B shape: (512, 512)


In [26]:
test_channel = clean_images[0][..., 0]

blurred_from_operator = operators.apply_motion_blur(test_channel, psf)

# For an image stored as rows, G = F @ B.T
blurred_from_B = test_channel @ B.T

max_difference = np.max(
    np.abs(blurred_from_operator - blurred_from_B)
)

print(f"Maximum absolute difference: {max_difference:.3e}")

np.testing.assert_allclose(
    blurred_from_B,
    blurred_from_operator,
    rtol=1e-10,
    atol=1e-10
)

print("Verification passed: B reproduces the reflective horizontal blur.")

Maximum absolute difference: 1.110e-15
Verification passed: B reproduces the reflective horizontal blur.


### 7. Zero-order Tikhonov regularization

As in the exercise, we use

$$
D=I.
$$

The Tikhonov functional for the RGB image is

$$
L_\lambda(f_{\mathrm{RGB}})
=
\|A_{\mathrm{RGB}}f_{\mathrm{RGB}}-g_{\mathrm{RGB}}\|_2^2
+
\lambda\|f_{\mathrm{RGB}}\|_2^2.
$$

We define the residual term

$$
\rho(\lambda)
=
\|A_{\mathrm{RGB}}f_{\lambda,\mathrm{RGB}}
-
g_{\mathrm{RGB}}\|_2^2,
$$

and the regularization term

$$
\eta(\lambda)
=
\|f_{\lambda,\mathrm{RGB}}\|_2^2.
$$

Therefore,

$$
L_\lambda(f_{\lambda,\mathrm{RGB}})
=
\rho(\lambda)
+
\lambda\eta(\lambda).
$$

Because the operator is block diagonal, the full problem separates into independent row-wise and channel-wise problems:

$$
L_\lambda(f_{\mathrm{RGB}})
=
\sum_{c\in\{R,G,B\}}
\sum_{r=1}^{512}
\left[
\|Bf_{r,c}-g_{r,c}\|_2^2
+
\lambda\|f_{r,c}\|_2^2
\right].
$$

In [27]:
def tikhonov_objective_rgb(candidate_rgb, observed_rgb, B, lam):
    """
    Compute the zero-order Tikhonov functional for an RGB image.

    Parameters
    ----------
    candidate_rgb : numpy.ndarray
        Candidate restored image with shape (height, width, 3).
    observed_rgb : numpy.ndarray
        Blurred/noisy RGB observation with shape (height, width, 3).
    B : numpy.ndarray
        Horizontal reflective blur matrix with shape (width, width).
    lam : float
        Tikhonov regularization parameter.

    Returns
    -------
    objective : float
        Tikhonov functional value.
    residual_term : float
        Data-fidelity term ||Af - g||_2^2.
    regularization_term : float
        Regularization term ||f||_2^2.
    """
    if candidate_rgb.shape != observed_rgb.shape:
        raise ValueError(
            "candidate_rgb and observed_rgb must have the same shape."
        )

    if candidate_rgb.ndim != 3 or candidate_rgb.shape[2] != 3:
        raise ValueError(
            "Images must have shape (height, width, 3)."
        )

    height, width, _ = candidate_rgb.shape

    if B.shape != (width, width):
        raise ValueError(
            "B must have shape (width, width)."
        )

    if lam < 0:
        raise ValueError("lam must be non-negative.")

    # For every channel, the image forward model is G = F @ B.T.
    predicted_rgb = np.empty_like(candidate_rgb, dtype=float)

    for channel in range(3):
        predicted_rgb[:, :, channel] = (
            candidate_rgb[:, :, channel] @ B.T
        )

    residual = predicted_rgb - observed_rgb

    residual_term = np.sum(residual**2)
    regularization_term = np.sum(candidate_rgb**2)

    objective = residual_term + lam * regularization_term

    return objective, residual_term, regularization_term

In [28]:
L_value, rho, eta = tikhonov_objective_rgb(
    candidate_rgb=clean_images[0],
    observed_rgb=processed_images[0][0],
    B=B,
    lam=0.01
)

print(f"L(lambda) = {L_value:.6e}")
print(f"Residual term rho = {rho:.6e}")
print(f"Regularization term eta = {eta:.6e}")

L(lambda) = 2.519304e+03
Residual term rho = 3.178350e+02
Regularization term eta = 2.201469e+05


### 8. Closed-form Tikhonov solution

Setting the derivative of the Tikhonov functional equal to zero gives

$$
A_{\mathrm{RGB}}^T
\left(
A_{\mathrm{RGB}}f_{\lambda,\mathrm{RGB}}
-
g_{\mathrm{RGB}}
\right)
+
\lambda f_{\lambda,\mathrm{RGB}}
=
0.
$$

Therefore,

$$
\left(
A_{\mathrm{RGB}}^T A_{\mathrm{RGB}}
+
\lambda I
\right)
f_{\lambda,\mathrm{RGB}}
=
A_{\mathrm{RGB}}^Tg_{\mathrm{RGB}}.
$$

Formally, the solution is

$$
f_{\lambda,\mathrm{RGB}}
=
\left(
A_{\mathrm{RGB}}^T A_{\mathrm{RGB}}
+
\lambda I
\right)^{-1}
A_{\mathrm{RGB}}^Tg_{\mathrm{RGB}}.
$$

Because the operator is block diagonal, each row and colour channel satisfies

$$
\left(
B^TB+\lambda I_{512}
\right)
f_{\lambda,r,c}
=
B^Tg_{r,c}.
$$

Thus,

$$
f_{\lambda,r,c}
=
\left(
B^TB+\lambda I_{512}
\right)^{-1}
B^Tg_{r,c}.
$$

For each value of $\lambda$, the matrix

$$
B^TB+\lambda I_{512}
$$

is factorized once and then used for all

$$
512\times3
$$

row-channel combinations.

In [29]:
def resolve_tikhonov(observed_rgb, B, lam):
    """
    Compute the zero-order Tikhonov reconstruction of an RGB image.

    The solution is obtained row by row and channel by channel from

        f_lambda = (B.T @ B + lam * I)^(-1) @ B.T @ g.

    Parameters
    ----------
    observed_rgb : numpy.ndarray
        Blurred/noisy RGB image with shape (height, width, 3).
    B : numpy.ndarray
        Reflective horizontal blur matrix with shape (width, width).
    lam : float
        Positive Tikhonov regularization parameter.

    Returns
    -------
    restored_rgb : numpy.ndarray
        Raw Tikhonov reconstruction with shape (height, width, 3).
    """
    observed_rgb = np.asarray(observed_rgb, dtype=float)

    if observed_rgb.ndim != 3 or observed_rgb.shape[2] != 3:
        raise ValueError(
            "observed_rgb must have shape (height, width, 3)."
        )

    if lam <= 0:
        raise ValueError("lam must be strictly positive.")

    height, width, n_channels = observed_rgb.shape

    if B.shape != (width, width):
        raise ValueError(
            f"B must have shape ({width}, {width}), "
            f"but has shape {B.shape}."
        )

    M_lambda = B.T @ B + lam * np.eye(width)

    restored_rgb = np.empty_like(observed_rgb, dtype=float)

    for channel in range(n_channels):
        # Row-wise right-hand side: G_c @ B
        rhs = observed_rgb[:, :, channel] @ B

        # Solve M_lambda @ X = rhs.T for all image rows simultaneously.
        restored_rgb[:, :, channel] = np.linalg.solve(
            M_lambda,
            rhs.T
        ).T

    return restored_rgb

### 9. L-curve parameter selection

For a set of logarithmically spaced parameters

$$
\lambda_j\in[10^{-5},10],
\qquad
j=1,\ldots,100,
$$

we compute

$$
f_{\lambda_j,\mathrm{RGB}},
\qquad
\rho(\lambda_j),
\qquad
\eta(\lambda_j).
$$

The L-curve is the log-log plot of

$$
\bigl(\rho(\lambda),\eta(\lambda)\bigr).
$$

Define

$$
\hat{\rho}=\log\rho,
\qquad
\hat{\eta}=\log\eta,
\qquad
s=\log\lambda.
$$

The curvature is

$$
\kappa(\lambda)
=
2
\frac{
\hat{\rho}'(s)\hat{\eta}''(s)
-
\hat{\rho}''(s)\hat{\eta}'(s)
}{
\left[
\hat{\rho}'(s)^2+
\hat{\eta}'(s)^2
\right]^{3/2}
}.
$$

The selected regularization parameter is

$$
\lambda_L
=
\arg\max_{\lambda_j}\kappa(\lambda_j).
$$

Since the clean RGB image is known, we also evaluate

$$
\mathrm{MSE}(\lambda)
=
\frac{1}{3N}
\left\|
f_{\lambda,\mathrm{RGB}}
-
f_{\mathrm{true},\mathrm{RGB}}
\right\|_2^2.
$$

The MSE is used only to assess reconstruction quality. The final RGB reconstruction is

$$
f_{\mathrm{final},\mathrm{RGB}}
=
f_{\lambda_L,\mathrm{RGB}}.
$$